# Step 3 — publish your labels as a data asset (`HCR-ROI-human-labeling`)

Run **after labeling**. The GUI wrote this session's labels to
`/root/capsule/scratch/labels/roi_qc_actions_<UTCstamp>.jsonl` — **`/scratch`, not `/results`,
because `/results` is ephemeral on a cloud workstation**. This captures that folder as a new
Code Ocean data asset tagged `HCR-ROI-human-labeling`, which the **training capsule** then
attaches (it merges all such label assets newest-wins).

In [ ]:
import os, datetime
from pathlib import Path
from lamf_analysis.code_ocean import code_ocean_utils as cou
from codeocean.data_asset import DataAssetParams, Source, CloudWorkstationSource

SUBJECT_IDS = ['790322']     # <-- the subject(s) you just labeled (for the name/tags)
REVIEWER    = 'anonymous'    # <-- your name
SUFFIX      = 'HCR-ROI-human-labeling'
LABELS_DIR  = '/root/capsule/scratch/labels'   # where label.sh wrote the labels (persistent)

files = sorted(Path(LABELS_DIR).glob('roi_qc_actions_*.jsonl'))
assert files, f'no label files in {LABELS_DIR} — did you label + write there?'
print(f'{len(files)} label file(s) to publish:'); [print('  ', f.name) for f in files]

In [ ]:
co = cou.get_co_client()
ts = datetime.datetime.utcnow().strftime('%Y-%m-%d_%H-%M-%S')
name = f"HCR_{'-'.join(SUBJECT_IDS)}_{SUFFIX}_{REVIEWER}_{ts}"

# Capture the /scratch labels folder from this cloud-workstation session.
params = DataAssetParams(
    name=name,
    mount=name,
    tags=['derived', 'HCR', SUFFIX] + [str(s) for s in SUBJECT_IDS],
    custom_metadata={'data level': 'derived', 'experiment type': 'HCR',
                     'subject id': ','.join(map(str, SUBJECT_IDS))},
    source=Source(cloud_workstation=CloudWorkstationSource(
        id=os.getenv('CO_COMPUTATION_ID'), path=LABELS_DIR)),
)
asset = co.data_assets.create_data_asset(params)
print('created data asset:', asset.id, '\n ', asset.name)